In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"

books = []

for page in range(1, 6):

    url = BASE_URL.format(page)

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    book_cards = soup.select("article.product_pod")

    print(f"Page {page}: {len(book_cards)} books found")

    for book in book_cards:

        # -------------------------
        # Title
        # -------------------------
        title = book.h3.a.get("title")

        # -------------------------
        # Price
        # -------------------------
        price = book.select_one(".price_color").get_text(strip=True)

        # -------------------------
        # Star rating
        # -------------------------
        rating_element = book.select_one(".star-rating")

        if rating_element:
            rating_classes = rating_element.get("class", [])
            star_rating = next(
                (x for x in rating_classes
                 if x in ["One", "Two", "Three", "Four", "Five"]),
                None
            )
        else:
            star_rating = None

        # -------------------------
        # Availability
        # -------------------------
        availability = book.select_one(".availability")

        if availability:
            availability = availability.get_text(" ", strip=True)
        else:
            availability = None

        # -------------------------
        # Book detail URL
        # -------------------------
        book_link = book.h3.a.get("href")
        detail_url = requests.compat.urljoin(url, book_link)

        # -------------------------
        # Get category
        # -------------------------
        detail_response = requests.get(detail_url, timeout=10)
        detail_response.raise_for_status()

        detail_soup = BeautifulSoup(
            detail_response.text,
            "html.parser"
        )

        breadcrumbs = detail_soup.select("ul.breadcrumb li a")

        if len(breadcrumbs) >= 3:
            category = breadcrumbs[2].get_text(strip=True)
        else:
            category = None

        books.append({
            "title": title,
            "price": price,
            "star_rating": star_rating,
            "availability": availability,
            "category": category
        })

print("Total books scraped:", len(books))

Page 1: 20 books found
Page 2: 20 books found
Page 3: 20 books found
Page 4: 20 books found
Page 5: 20 books found
Total books scraped: 100


In [3]:
df_raw = pd.DataFrame(books)

df_raw.head()

,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History


In [4]:
print(df_raw.shape)
print(df_raw.columns)

(100, 5)
Index(['title', 'price', 'star_rating', 'availability', 'category'], dtype='object')


In [6]:
print(df_raw["category"].value_counts)

<bound method IndexOpsMixin.value_counts of 0                 Poetry
1     Historical Fiction
2                Fiction
3                Mystery
4                History
             ...        
95        Sequential Art
96        Food and Drink
97         Add a comment
98       Science Fiction
99            Nonfiction
Name: category, Length: 100, dtype: object>


In [7]:
df_raw["category"].nunique()

29

Clean the data

In [8]:
def clean_price(value):
    try:
        return float(
            str(value)
            .replace("Â£", "")
            .strip()
        )
    except (ValueError, TypeError):
        return None


df_raw["price_gbp"] = df_raw["price"].apply(clean_price)

In [9]:
df_raw[['price','price_gbp']].head()

,price,price_gbp
0,Â£51.77,51.77
1,Â£53.74,53.74
2,Â£50.10,50.10
3,Â£47.82,47.82
4,Â£54.23,54.23


In [10]:
df_raw.dtypes

title            object
price            object
star_rating      object
availability     object
category         object
price_gbp       float64
dtype: object

Convert star rating to integer

In [11]:
rating_map = {
    "One" : 1,
    "Two" : 2,
    "Three" : 3,
    "Four" : 4,
    "Five" :5
}

df_raw['rating'] = df_raw['star_rating'].map(rating_map)

In [12]:
df_raw[['star_rating','rating']].head()

,star_rating,rating
0,Three,3
1,One,1
2,One,1
3,Four,4
4,Five,5


In [13]:
df_raw.dtypes

title            object
price            object
star_rating      object
availability     object
category         object
price_gbp       float64
rating            int64
dtype: object

Parse availability

In [14]:
def parse_stock(value):

    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    if "in stock" in value:
        return True

    if "out of stock" in value:
        return False

    return None


df_raw["in_stock"] = df_raw["availability"].apply(parse_stock)

In [16]:
df_raw[['availability','in_stock']].head()

,availability,in_stock
0,In stock,True
1,In stock,True
2,In stock,True
3,In stock,True
4,In stock,True


In [17]:
df_raw.head()

,title,price,star_rating,availability,category,price_gbp,rating,in_stock
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry,51.77,3,True
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction,53.74,1,True
2,Soumission,Â£50.10,One,In stock,Fiction,50.10,1,True
3,Sharp Objects,Â£47.82,Four,In stock,Mystery,47.82,4,True
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History,54.23,5,True


In [19]:
df_raw.isnull().sum()

title           0
price           0
star_rating     0
availability    0
category        0
price_gbp       0
rating          0
in_stock        0
dtype: int64

Handle parsing failures

In [20]:
numeric_columns = ['price_gbp','rating']

for col in numeric_columns:
    median_value = df_raw[col].median()

    df_raw[col] =df_raw[col].fillna(median_value)

In [21]:
required_columns = [
    "title",
    "category",
    "in_stock"
]

df_clean = df_raw.dropna(
    subset=required_columns
).copy()

In [22]:
df_clean.dtypes

title            object
price            object
star_rating      object
availability     object
category         object
price_gbp       float64
rating            int64
in_stock           bool
dtype: object

In [23]:
df_clean["price_gbp"] = df_clean["price_gbp"].astype(float)

df_clean["rating"] = df_clean["rating"].astype(int)

df_clean["in_stock"] = df_clean["in_stock"].astype(bool)

In [24]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         100 non-null    object 
 1   price         100 non-null    object 
 2   star_rating   100 non-null    object 
 3   availability  100 non-null    object 
 4   category      100 non-null    object 
 5   price_gbp     100 non-null    float64
 6   rating        100 non-null    int32  
 7   in_stock      100 non-null    bool   
dtypes: bool(1), float64(1), int32(1), object(5)
memory usage: 5.3+ KB


Fixed GBP → INR conversion

In [25]:
GBP_TO_INR = 105.50

df_clean['price_inr'] = ( df_clean["price_gbp"] * GBP_TO_INR).round(2)

In [26]:
df_clean[['title','price_gbp','price_inr']].head()

,title,price_gbp,price_inr
0,A Light in the Attic,51.77,5461.74
1,Tipping the Velvet,53.74,5669.57
2,Soumission,50.10,5285.55
3,Sharp Objects,47.82,5045.01
4,Sapiens: A Brief History of Humankind,54.23,5721.26


In [27]:
df_clean.head()

,title,price,star_rating,availability,category,price_gbp,rating,in_stock,price_inr
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry,51.77,3,True,5461.74
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction,53.74,1,True,5669.57
2,Soumission,Â£50.10,One,In stock,Fiction,50.10,1,True,5285.55
3,Sharp Objects,Â£47.82,Four,In stock,Mystery,47.82,4,True,5045.01
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History,54.23,5,True,5721.26


Create the normalized SQLite database

In [28]:
import sqlite3

In [29]:

conn = sqlite3.connect("books_catalogue.db")

cursor =  conn.cursor()

cursor.execute("""DROP TABLE IF EXISTS categories""")

cursor.execute("""DROP TABLE IF EXISTS books""")

conn.commit()

In [30]:
conn = sqlite3.connect("books_catalogue.db")

cursor =  conn.cursor()

cursor.execute(""" CREATE TABLE IF NOT EXISTS categories (category_id INTEGER PRIMARY KEY AUTOINCREMENT , category_name TEXT UNIQUE NOT NULL)""")

cursor.execute("""CREATE TABLE IF NOT EXISTS books(book_id INTEGER PRIMARY KEY AUTOINCREMENT , title TEXT NOT NULL , price_gbp REAL NOT NULL , price_inr REAL NOT NULL ,  rating INTEGER NOT NULL ,in_stock INTEGER NOT NULL, category_id INTEGER NOT NULL , FOREIGN KEY(category_id) REFERENCES categories(category_id))""")

conn.commit()

Insert categories

In [31]:
categories_df = (
    df_clean[['category']]
    .drop_duplicates()
    .sort_values('category')
    .reset_index(drop=True)
)

categories_df

,category
0,Add a comment
1,Art
2,Business
3,Childrens
4,Contemporary
5,Default
6,Fantasy
7,Fiction
8,Food and Drink
9,Health


In [32]:
categories_df = (df_clean[['category']].drop_duplicates().rename(columns = {"category" : "category_name"}))



In [33]:
categories_df.to_sql(
    "categories",
    conn,
    if_exists = "append",
    index = False,
    dtype={'category':'TEXT'}
)

29

In [34]:
category_lookup  = pd.read_sql(
    "SELECT category_id , category_name FROM categories",
    conn
)

category_lookup.head()

,category_id,category_name
0,1,Poetry
1,2,Historical Fiction
2,3,Fiction
3,4,Mystery
4,5,History


Create the book DataFrame with category IDs

In [35]:
books_df  = df_clean.merge(
    category_lookup,
    left_on  = 'category',
    right_on  = 'category_name',
    how ='left'
)

In [36]:
books_to_insert  = books_df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_id"
    ]
].copy()

In [38]:
books_to_insert['in_stock']  = (
    books_to_insert["in_stock"].astype(int)
)

In [39]:
books_to_insert.to_sql(
    "books",
    conn,
    if_exists  = "append",
    index  = False
)

conn.commit()

Verify the database

In [40]:
pd.read_sql(
    "SELECT COUNT(*) AS book_count FROM books",
    conn
)

,book_count
0,100


In [41]:
pd.read_sql(
    "select count(*) as category_count from categories",
    conn
)

,category_count
0,29


Query 1 — SELECT + WHERE

books costing more than £40:

In [42]:
query1 = """
SELECT
    title,
    price_gbp,
    rating,
    in_stock
FROM books
WHERE price_gbp > 40
"""

result1 = pd.read_sql(query1, conn)

result1.head(10)

,title,price_gbp,rating,in_stock
0,A Light in the Attic,51.77,3,1
1,Tipping the Velvet,53.74,1,1
2,Soumission,50.10,1,1
3,Sharp Objects,47.82,4,1
4,Sapiens: A Brief History of Humankind,54.23,5,1
5,The Black Maria,52.15,1,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5,1
7,Our Band Could Be Your Life: Scenes from the A...,57.25,3,1
8,Libertarianism for Beginners,51.33,2,1
9,It's Only the Himalayas,45.17,2,1


In [43]:
result1.to_csv("query1_output.csv", index=False)

Query 2 — ORDER BY + LIMIT

Find the 10 most expensive books:

In [44]:
query2 = """
SELECT
    title,
    price_gbp,
    price_inr
FROM books
ORDER BY price_gbp DESC
LIMIT 10
"""

result2 = pd.read_sql(query2, conn)

result2

,title,price_gbp,price_inr
0,The Death of Humanity: and the Case for Life,58.11,6130.60
1,Slow States of Collapse: Poems,57.31,6046.20
2,Our Band Could Be Your Life: Scenes from the A...,57.25,6039.88
3,The Past Never Ends,56.50,5960.75
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41,5951.25
5,Masks and Shadows,56.40,5950.20
6,The Secret of Dreadwillow Carse,56.13,5921.72
7,The Electric Pencil: Drawings from Inside Stat...,56.06,5914.33
8,Birdsong: A Story in Pictures,54.64,5764.52
9,Sapiens: A Brief History of Humankind,54.23,5721.26


In [45]:
result2.to_csv("query2_output.csv", index=False)

Query 3 — DISTINCT

List all unique categories:

In [46]:
query3 = """
SELECT DISTINCT category_name
FROM categories
ORDER BY category_name
"""

result3 = pd.read_sql(query3, conn)

result3

,category_name
0,Add a comment
1,Art
2,Business
3,Childrens
4,Contemporary
5,Default
6,Fantasy
7,Fiction
8,Food and Drink
9,Health


In [47]:
result3.to_csv("query3_output.csv", index=False)

Query 4 — BETWEEN

Find books priced between £20 and £30:

In [48]:
query4 = """
SELECT
    title,
    price_gbp,
    rating
FROM books
WHERE price_gbp BETWEEN 20 AND 30
ORDER BY price_gbp
"""

result4 = pd.read_sql(query4, conn)

result4.head(10)

,title,price_gbp,rating
0,The Inefficiency Assassin: Time Management Tac...,20.59,5
1,Shakespeare's Sonnets,20.66,4
2,In the Country We Love: My Family Divided,22.00,4
3,America's Cradle of Quarterbacks: Western Penn...,22.50,3
4,The Boys in the Boat: Nine Americans and Their...,22.60,4
5,The Requiem Red,22.65,1
6,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,5
7,The Elephant Tree,23.82,5
8,Olio,23.88,1
9,The Mindfulness and Acceptance Workbook for An...,23.89,4


In [49]:
result4.to_csv("query4_output.csv", index=False)

Query 5 — JOIN

In [50]:
query5 = """
SELECT
    b.title,
    c.category_name,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
WHERE b.rating >= 4
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10
"""

result5 = pd.read_sql(query5, conn)

result5

,title,category_name,price_gbp,price_inr,rating,in_stock
0,Sapiens: A Brief History of Humankind,History,54.23,5721.26,5,1
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,52.29,5516.60,5,1
2,"We Love You, Charlie Freeman",Fiction,50.27,5303.48,5,1
3,Private Paris (Private #10),Fiction,47.61,5022.85,5,1
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,40.30,4251.65,5,1
5,Join,Science Fiction,35.67,3763.19,5,1
6,Rip it Up and Start Again,Music,35.02,3694.61,5,1
7,Black Dust,Romance,34.53,3642.92,5,1
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,32.24,3401.32,5,1
9,Chase Me (Paris Nights #2),Romance,25.27,2665.98,5,1


In [51]:
result5.to_csv("query5_output.csv", index=False)

In [52]:
query_join = """
SELECT
    category_name,
    title,
    price_gbp,
    rating,
    in_stock
FROM (
    SELECT
        c.category_name,
        b.title,
        b.price_gbp,
        b.rating,
        b.in_stock,
        ROW_NUMBER() OVER (
            PARTITION BY c.category_name
            ORDER BY b.rating DESC, b.price_gbp DESC
        ) AS rn
    FROM books b
    JOIN categories c
        ON b.category_id = c.category_id
)
WHERE rn <= 10
ORDER BY category_name, rating DESC
"""

result_join_sql = pd.read_sql(query_join, conn)

result_join_sql.head(20)

,category_name,title,price_gbp,rating,in_stock
0,Add a comment,The Mindfulness and Acceptance Workbook for An...,23.89,4,1
1,Add a comment,The Art Forger,40.76,3,1
2,Add a comment,On a Midnight Clear,14.07,3,1
3,Add a comment,Judo: Seven Steps to Black Belt (an Introducto...,53.90,2,1
4,Add a comment,The Torch Is Passed: A Harding Family Story,19.09,1,1
5,Art,Wall and Piece,44.18,4,1
6,Business,The Dirty Little Secrets of Getting Your Dream...,33.34,4,1
7,Childrens,Birdsong: A Story in Pictures,54.64,3,1
8,Childrens,The Secret of Dreadwillow Carse,56.13,1,1
9,Childrens,The Bear and the Piano,36.89,1,1


Reproduce the JOIN using pandas

In [53]:
books_memory = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_memory = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

In [54]:
pandas_join = pd.merge(
    books_memory,
    categories_memory,
    on="category_id",
    how="inner"
)

In [55]:
pandas_join = pandas_join[
    [
        "category_name",
        "title",
        "price_gbp",
        "rating",
        "in_stock"
    ]
]

In [56]:
pandas_join = pandas_join.sort_values(
    ["category_name", "rating", "price_gbp"],
    ascending=[True, False, False]
)

pandas_join["rank"] = (
    pandas_join
    .groupby("category_name")
    .cumcount() + 1
)

result_join_pandas = (
    pandas_join[
        pandas_join["rank"] <= 10
    ]
    .drop(columns="rank")
    .reset_index(drop=True)
)

Compare SQL JOIN vs Pandas JOIN

In [57]:
sql_compare = result_join_sql.reset_index(drop=True)

pandas_compare = result_join_pandas.reset_index(drop=True)

# Make sure column ordering is identical
pandas_compare = pandas_compare[
    sql_compare.columns
]

In [58]:
print(
    sql_compare.equals(pandas_compare)
)

True


In [59]:
print("SQL rows:", len(sql_compare))
print("Pandas rows:", len(pandas_compare))

SQL rows: 94
Pandas rows: 94


In [60]:
comparison = pd.concat(
    [
        sql_compare.add_suffix("_sql"),
        pandas_compare.add_suffix("_pandas")
    ],
    axis=1
)

comparison.head()

,category_name_sql,title_sql,price_gbp_sql,rating_sql,in_stock_sql,category_name_pandas,title_pandas,price_gbp_pandas,rating_pandas,in_stock_pandas
0,Add a comment,The Mindfulness and Acceptance Workbook for An...,23.89,4,1,Add a comment,The Mindfulness and Acceptance Workbook for An...,23.89,4,1
1,Add a comment,The Art Forger,40.76,3,1,Add a comment,The Art Forger,40.76,3,1
2,Add a comment,On a Midnight Clear,14.07,3,1,Add a comment,On a Midnight Clear,14.07,3,1
3,Add a comment,Judo: Seven Steps to Black Belt (an Introducto...,53.90,2,1,Add a comment,Judo: Seven Steps to Black Belt (an Introducto...,53.90,2,1
4,Add a comment,The Torch Is Passed: A Harding Family Story,19.09,1,1,Add a comment,The Torch Is Passed: A Harding Family Story,19.09,1,1


In [61]:
queries = {
    "query1_select_where": query1,
    "query2_order_limit": query2,
    "query3_distinct": query3,
    "query4_between": query4,
    "query5_join": query5,
    "query_join_top_per_category": query_join
}

In [62]:
with open("sql_queries.txt", "w", encoding="utf-8") as f:

    for name, query in queries.items():

        f.write("=" * 60 + "\n")
        f.write(name + "\n")
        f.write("=" * 60 + "\n")
        f.write(query.strip() + "\n\n")

In [63]:
final_df = df_clean[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
].copy()

final_df.head()

,title,price_gbp,price_inr,rating,in_stock,category
0,A Light in the Attic,51.77,5461.74,3,True,Poetry
1,Tipping the Velvet,53.74,5669.57,1,True,Historical Fiction
2,Soumission,50.10,5285.55,1,True,Fiction
3,Sharp Objects,47.82,5045.01,4,True,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5721.26,5,True,History


In [64]:
final_df.to_csv(
    "books_cleaned.csv",
    index=False
)

In [65]:
conn.close()

In [66]:
print("\n========== PIPELINE SUMMARY ==========")
print(f"Books scraped: {len(df_raw)}")
print(f"Books after cleaning: {len(df_clean)}")
print(f"Categories: {df_clean['category'].nunique()}")

print("\nData types:")
print(df_clean[
    ["price_gbp", "rating", "in_stock", "price_inr"]
].dtypes)

print("\nJOIN comparison:")
print("SQL rows:", len(result_join_sql))
print("Pandas rows:", len(result_join_pandas))
print("Results match:", sql_compare.equals(pandas_compare))


========== PIPELINE SUMMARY ==========
Books scraped: 100
Books after cleaning: 100
Categories: 29

Data types:
price_gbp    float64
rating         int32
in_stock        bool
price_inr    float64
dtype: object

JOIN comparison:
SQL rows: 94
Pandas rows: 94
Results match: True
